## Import Library & API

In [38]:
import pandas as pd
import pandas_ta as ta
from pandas.tseries.offsets import BusinessDay
import pandas_market_calendars as mcal
import pywt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.ardl import ARDL, ardl_select_order, UECM
from statsmodels.stats.diagnostic import het_arch
from statsmodels.tools.sm_exceptions import ValueWarning
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
import fredapi as fa
from datetime import date
from twelvedata import TDClient
import vectorbt as vbt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Input, LSTM, BatchNormalization, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.losses import Huber
import time
import re
import os

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore', ValueWarning)

os.chdir('/Users/fulinq/Documents/KMITL/FinancialEngineering/Y4/Y4T1/PROJECT/ARDL-ECM/Code/Gold/Finalized/V.1')

In [39]:
fred = fa.Fred(api_key='c948956426006ca126a2dd3bd1f07cee')
td = TDClient(apikey='aa61c51218c248698467af34d09b9d46')

## Data Retrieve ##

In [40]:
def fetch_fred(fred_client, series_id, col_name, percent = False, save_csv=False):
    df = fred_client.get_series(series_id)
    df.index = pd.to_datetime(df.index) 
    print(f'NaN value before processing: {df.isna().sum()}')
    df = df.ffill()
    print(f'NaN value after processing: {df.isna().sum()}')
    df.rename(col_name, inplace=True)
    print(f'Total records for {col_name}: {len(df)}')
    print(f'start date: {df.index.min()}')
    print(f'end date: {df.index.max()}')
    
    if percent:
        df = df.mul(0.01)
        print(f'Total records for {col_name} in percent: {len(df)}')
    
    if save_csv:
        filename = f"all_{col_name.lower()}_data_fred.csv"
        df.to_csv(filename)
        print(f"FRED data saved to {filename}")
    
    return pd.DataFrame(df)

def chow_lin_disaggregate(y_low: pd.Series, X_high: pd.DataFrame,
                          agg_method: str = 'sum', rho: float = None) -> tuple:
    y_low = y_low.dropna().copy()
    X_high = X_high.dropna().copy()
    n_high_per_low = 3  # Quarterly -> Monthly = 3 เดือนต่อไตรมาส

    # หาช่วงเวลาที่ซ้อนทับกัน (Overlapping period)
    quarters = y_low.index
    months = X_high.index
    min_date = max(quarters.min(), months.min().to_period('Q').to_timestamp())
    max_date = min(quarters.max(), months.max().to_period('Q').to_timestamp())

    y_low = y_low[(y_low.index >= min_date) & (y_low.index <= max_date)]
    
    # ปรับช่วงเวลาของ Monthly ให้ครอบคลุม Quarterly พอดี
    month_start = y_low.index.min()
    month_end = (y_low.index.max() + pd.offsets.QuarterEnd()).to_period('M').to_timestamp()
    X_high = X_high[(X_high.index >= month_start) & (X_high.index <= month_end)]

    n_low = len(y_low)
    n_high = n_low * n_high_per_low
    X_high = X_high.iloc[:n_high] # ตัดส่วนเกินออก

    # Build aggregation matrix C (Matrix สำหรับแปลงรายเดือนกลับเป็นไตรมาส)
    C = np.zeros((n_low, n_high))
    for i in range(n_low):
        start_col = i * n_high_per_low
        end_col = start_col + n_high_per_low
        if agg_method == 'sum': # สำหรับ Flow variable เช่น GDP
            C[i, start_col:end_col] = 1.0
        elif agg_method == 'mean': # สำหรับ Stock variable
            C[i, start_col:end_col] = 1.0 / n_high_per_low
        else:
            C[i, end_col - 1] = 1.0

    # Prepare X matrix
    X = X_high.values
    if X.ndim == 1: X = X.reshape(-1, 1)
    X = np.column_stack([np.ones(n_high), X]) # เพิ่ม Intercept

    # OLS เบื้องต้นเพื่อหาค่า Rho (Autocorrelation coefficient)
    X_low = C @ X
    y = y_low.values.flatten()
    beta_ols = np.linalg.lstsq(X_low, y, rcond=None)[0]
    u_low = y - X_low @ beta_ols

    if rho is None: # ถ้าไม่ได้กำหนดมา ให้คำนวณจาก Residuals
        if len(u_low) > 1:
            rho = np.corrcoef(u_low[:-1], u_low[1:])[0, 1]
            rho = np.clip(rho, -0.99, 0.99)
        else:
            rho = 0.0

    # GLS Estimation (พระเอกของงาน)
    # สร้าง Covariance Matrix V ตามโครงสร้าง AR(1)
    V = np.zeros((n_high, n_high))
    for i in range(n_high):
        for j in range(n_high):
            V[i, j] = rho ** abs(i - j)

    V_low = C @ V @ C.T
    try:
        V_low_inv = np.linalg.inv(V_low)
    except:
        V_low_inv = np.linalg.pinv(V_low)

    # คำนวณ Beta ด้วย GLS
    XVX = X_low.T @ V_low_inv @ X_low
    XVy = X_low.T @ V_low_inv @ y
    try:
        beta_gls = np.linalg.solve(XVX, XVy)
    except:
        beta_gls = np.linalg.lstsq(XVX, XVy, rcond=None)[0]

    # คำนวณค่าพยากรณ์และกระจาย Error (Distribute residuals)
    p_high = X @ beta_gls
    u_low_gls = y - X_low @ beta_gls
    VCt = V @ C.T
    
    try:
        dist_matrix = VCt @ np.linalg.inv(V_low)
    except:
        dist_matrix = VCt @ np.linalg.pinv(V_low)

    y_high = p_high + dist_matrix @ u_low_gls # ผลลัพธ์สุดท้าย

    result = pd.Series(y_high, index=X_high.index, name='GDP_Monthly_ChowLin')
    return result, beta_gls, rho

In [41]:
gold = vbt.YFData.download("GC=F", start="2006-01-01", interval="1mo").get()
gold = pd.DataFrame(gold)
gold.columns = gold.columns.str.lower()
gold.index = pd.to_datetime(gold.index).tz_localize(None)
gold = gold.sort_index()
gold = gold.drop(columns=['dividends', 'stock splits'])
gold.to_csv('all_gold_data.csv')
gold = gold.get('close')
gold

Date
2006-02-01 05:00:00     561.599976
2006-03-01 05:00:00     581.799988
2006-04-01 05:00:00     651.799988
2006-05-01 04:00:00     642.500000
2006-06-01 04:00:00     613.500000
                          ...     
2025-10-01 04:00:00    3982.199951
2025-11-01 04:00:00    4218.299805
2025-12-01 05:00:00    4325.600098
2026-01-01 05:00:00    4713.899902
2026-02-19 05:00:00    5020.299805
Name: close, Length: 207, dtype: float64

In [42]:
dollar_index = fetch_fred(fred, series_id='DTWEXBGS', col_name='Dollar Index')
dollar_index

NaN value before processing: 207
NaN value after processing: 0
Total records for Dollar Index: 5250
start date: 2006-01-02 00:00:00
end date: 2026-02-13 00:00:00


,Dollar Index
2006-01-02,101.4155
2006-01-03,100.7558
2006-01-04,100.2288
2006-01-05,100.2992
2006-01-06,100.0241
...,...
2026-02-09,117.6392
2026-02-10,117.5216
2026-02-11,117.4601
2026-02-12,117.5376


In [43]:
ppi = fetch_fred(fred, series_id='PPIACO', col_name='PPI')
ppi

NaN value before processing: 0
NaN value after processing: 0
Total records for PPI: 1356
start date: 1913-01-01 00:00:00
end date: 2025-12-01 00:00:00


,PPI
1913-01-01,12.100
1913-02-01,12.000
1913-03-01,12.000
1913-04-01,12.000
1913-05-01,11.900
...,...
2025-08-01,262.110
2025-09-01,262.094
2025-10-01,260.724
2025-11-01,261.358


In [44]:
fed_fund = fetch_fred(fred, series_id='FEDFUNDS', col_name='Federal Fund Rate', percent=True)
fed_fund

NaN value before processing: 0
NaN value after processing: 0
Total records for Federal Fund Rate: 859
start date: 1954-07-01 00:00:00
end date: 2026-01-01 00:00:00
Total records for Federal Fund Rate in percent: 859


,Federal Fund Rate
1954-07-01,0.0080
1954-08-01,0.0122
1954-09-01,0.0107
1954-10-01,0.0085
1954-11-01,0.0083
...,...
2025-09-01,0.0422
2025-10-01,0.0409
2025-11-01,0.0388
2025-12-01,0.0372


In [45]:
vix = fetch_fred(fred, series_id='VIXCLS', percent=True,col_name='VIX')
vix['VIX'] = vix['VIX'].mul(1 / np.sqrt(12))
vix

NaN value before processing: 301
NaN value after processing: 0
Total records for VIX: 9427
start date: 1990-01-02 00:00:00
end date: 2026-02-18 00:00:00
Total records for VIX in percent: 9427


,VIX
1990-01-02,0.049768
1990-01-03,0.052510
1990-01-04,0.055483
1990-01-05,0.058053
1990-01-08,0.058486
...,...
2026-02-12,0.060102
2026-02-13,0.059467
2026-02-16,0.061199
2026-02-17,0.058572


In [46]:
unemploy = fetch_fred(fred, series_id='ICSA', col_name='ISCA') #Initial Claims
unemploy

NaN value before processing: 0
NaN value after processing: 0
Total records for ISCA: 3085
start date: 1967-01-07 00:00:00
end date: 2026-02-14 00:00:00


,ISCA
1967-01-07,208000.0
1967-01-14,207000.0
1967-01-21,217000.0
1967-01-28,204000.0
1967-02-04,216000.0
...,...
2026-01-17,210000.0
2026-01-24,209000.0
2026-01-31,232000.0
2026-02-07,229000.0


In [47]:
ip = fetch_fred(fred, series_id='INDPRO', col_name='IP')
ip

NaN value before processing: 0
NaN value after processing: 0
Total records for IP: 1285
start date: 1919-01-01 00:00:00
end date: 2026-01-01 00:00:00


,IP
1919-01-01,4.8739
1919-02-01,4.6585
1919-03-01,4.5238
1919-04-01,4.6046
1919-05-01,4.6315
...,...
2025-09-01,101.7059
2025-10-01,101.2570
2025-11-01,101.3775
2025-12-01,101.6296


In [48]:
gdp = fetch_fred(fred, series_id='GDP', col_name='GDP')
gdp

NaN value before processing: 4
NaN value after processing: 4
Total records for GDP: 319
start date: 1946-01-01 00:00:00
end date: 2025-07-01 00:00:00


,GDP
1946-01-01,NaN
1946-04-01,NaN
1946-07-01,NaN
1946-10-01,NaN
1947-01-01,243.164
...,...
2024-07-01,29511.664
2024-10-01,29825.182
2025-01-01,30042.113
2025-04-01,30485.729


In [49]:
y_target = gdp['GDP']
X_indicator = ip[['IP']]

gdp_monthly_gls, beta, rho = chow_lin_disaggregate(y_low=y_target, X_high=X_indicator, agg_method='sum', rho=None)
print("Estimated Rho (Autocorrelation):", rho)
gdp = gdp_monthly_gls.copy()
gdp_monthly_gls

Estimated Rho (Autocorrelation): 0.99


1947-01-01       77.774623
1947-02-01       80.692609
1947-03-01       84.696768
1947-04-01       78.954071
1947-05-01       82.894270
                  ...     
2025-05-01    10144.515129
2025-06-01    10243.720934
2025-07-01    10350.652186
2025-08-01    10374.011335
2025-09-01    10373.363479
Name: GDP_Monthly_ChowLin, Length: 945, dtype: float64

In [50]:
fed_balance = fetch_fred(fred, series_id='WALCL', col_name='Fed Balance Sheet') #Federal Reserve Total Assets
fed_balance

NaN value before processing: 0
NaN value after processing: 0
Total records for Fed Balance Sheet: 1209
start date: 2002-12-18 00:00:00
end date: 2026-02-11 00:00:00


,Fed Balance Sheet
2002-12-18,719542.0
2002-12-25,732059.0
2003-01-01,730994.0
2003-01-08,723762.0
2003-01-15,720074.0
...,...
2026-01-14,6581700.0
2026-01-21,6584580.0
2026-01-28,6587568.0
2026-02-04,6605909.0


In [51]:
# 1. organize data
realtime_data = {
    'gold': gold,
    'dollar_index': dollar_index,
    'vix': vix,
    'fed_rate': fed_fund,
    'fed_balance': fed_balance,
    'labor_claims': unemploy
}

lagged_data = {
    'ip': ip,
    'gdp': gdp,
    'ppi': ppi
}

# 2. resample & rename
monthly_dfs = []

# process real-time
for name, data in realtime_data.items():
    # FIX: force rename for both Series and DataFrame to match the key (lowercase)
    if isinstance(data, pd.DataFrame):
        data = data.iloc[:, 0].to_frame(name)
    else:
        data = data.to_frame(name)
    
    if name in ['labor_claims', 'vix']:
        monthly_dfs.append(data.resample('ME').mean())
    else:
        monthly_dfs.append(data.resample('ME').last())

# process lagged
for name, data in lagged_data.items():
    if isinstance(data, pd.DataFrame):
        data = data.iloc[:, 0].to_frame(name)
    else:
        data = data.to_frame(name)
    monthly_dfs.append(data.resample('ME').last())

# 3. merge
df_final = pd.concat(monthly_dfs, axis=1)

# 4. handle lag (shift)
vars_to_shift = ['ip', 'ppi']
for col in vars_to_shift:
    df_final[col] = df_final[col].shift(1)
df_final['gdp'] = df_final['gdp'].shift(4)

# 5. target variable
df_final['target_gold'] = df_final['gold'].shift(-1)

# 6. feature selection
features = [
    'gold', 'dollar_index', 'vix', 'fed_rate', 
    'fed_balance', 'labor_claims', 
    'ip', 'gdp','ppi'
]

df_model = df_final[features + ['target_gold']].dropna()

# check
print(f"data range: {df_model.index.min().date()} to {df_model.index.max().date()}")
print(df_model.columns)
df_model

data range: 2006-02-28 to 2026-01-31
Index(['gold', 'dollar_index', 'vix', 'fed_rate', 'fed_balance',
       'labor_claims', 'ip', 'gdp', 'ppi', 'target_gold'],
      dtype='object')


,gold,dollar_index,vix,fed_rate,fed_balance,labor_claims,ip,gdp,ppi,target_gold
2006-02-28,561.599976,99.7695,0.035934,0.0449,840555.0,290750.0,98.1999,4387.341134,164.300,581.799988
2006-03-31,581.799988,100.5600,0.033757,0.0459,833675.0,301750.0,98.2413,4449.366137,161.800,651.799988
2006-04-30,651.799988,98.1412,0.034277,0.0479,844572.0,303600.0,98.4628,4487.496729,162.200,642.500000
2006-05-31,642.500000,97.7705,0.041702,0.0494,851580.0,332750.0,98.7618,4515.040605,164.300,613.500000
2006-06-30,613.500000,98.2483,0.048840,0.0499,844436.0,305500.0,98.7869,4531.215921,165.800,634.200012
...,...,...,...,...,...,...,...,...,...,...
2025-09-30,3840.800049,120.1368,0.045579,0.0422,6608395.0,234750.0,101.6247,10144.515129,262.110,3982.199951
2025-10-31,3982.199951,121.3859,0.052211,0.0409,6587034.0,226750.0,101.7059,10243.720934,262.094,4218.299805
2025-11-30,4218.299805,121.0527,0.057070,0.0388,6552419.0,217600.0,101.2570,10350.652186,260.724,4325.600098
2025-12-31,4325.600098,119.7456,0.044623,0.0372,6640618.0,219000.0,101.3775,10374.011335,261.358,4713.899902


In [52]:
df_model.to_csv('gold_price_model_data.csv')

In [53]:
df_ret = pd.DataFrame()
cols_to_transform = ['gold', 'gdp', 'ip', 'ppi','dollar_index', 'labor_claims', 'fed_balance'] # ไม่เอา IP, PPI ตามแผน Core Model
cols_not_to_transform = ['fed_rate', 'vix'] # ตัวแปรที่ไม่ทำ log return
for col in cols_to_transform:
    if col in df_model.columns:
        df_ret[f'{col}_ret'] = np.log(df_model[col]).diff()
for col in cols_not_to_transform:
    if col in df_model.columns:
        df_ret[f'{col}_change'] = df_model[col].diff()
    
df_ret.dropna(inplace=True)
df_ret

,gold_ret,gdp_ret,ip_ret,ppi_ret,dollar_index_ret,labor_claims_ret,fed_balance_ret,fed_rate_change,vix_change
2006-03-31,0.035337,0.014038,0.000422,-0.015333,0.007892,0.037135,-0.008219,0.0010,-0.002177
2006-04-30,0.113611,0.008533,0.002252,0.002469,-0.024347,0.006112,0.012986,0.0020,0.000520
2006-05-31,-0.014371,0.006119,0.003032,0.012864,-0.003784,0.091680,0.008263,0.0015,0.007425
2006-06-30,-0.046187,0.003576,0.000254,0.009088,0.004875,-0.085442,-0.008424,0.0005,0.007138
2006-07-31,0.033184,0.004775,0.003161,0.001808,-0.002535,0.042614,-0.003421,0.0025,-0.004910
...,...,...,...,...,...,...,...,...,...
2025-09-30,0.100460,0.004646,-0.002646,-0.000946,-0.000594,0.020442,0.000759,-0.0011,0.000113
2025-10-31,0.036154,0.009732,0.000799,-0.000061,0.010344,-0.034673,-0.003238,-0.0013,0.006632
2025-11-30,0.057598,0.010385,-0.004423,-0.005241,-0.002749,-0.041190,-0.005269,-0.0021,0.004858
2025-12-31,0.025119,0.002254,0.001189,0.002429,-0.010856,0.006413,0.013371,-0.0016,-0.012447


## Data Preparation ##

In [54]:
df_model = pd.read_csv('gold_price_model_data.csv', index_col=0, parse_dates=True)
df_model

,gold,dollar_index,vix,fed_rate,fed_balance,labor_claims,ip,gdp,ppi,target_gold
2006-02-28,561.599976,99.7695,0.035934,0.0449,840555.0,290750.0,98.1999,4387.341134,164.300,581.799988
2006-03-31,581.799988,100.5600,0.033757,0.0459,833675.0,301750.0,98.2413,4449.366137,161.800,651.799988
2006-04-30,651.799988,98.1412,0.034277,0.0479,844572.0,303600.0,98.4628,4487.496729,162.200,642.500000
2006-05-31,642.500000,97.7705,0.041702,0.0494,851580.0,332750.0,98.7618,4515.040605,164.300,613.500000
2006-06-30,613.500000,98.2483,0.048840,0.0499,844436.0,305500.0,98.7869,4531.215921,165.800,634.200012
...,...,...,...,...,...,...,...,...,...,...
2025-09-30,3840.800049,120.1368,0.045579,0.0422,6608395.0,234750.0,101.6247,10144.515129,262.110,3982.199951
2025-10-31,3982.199951,121.3859,0.052211,0.0409,6587034.0,226750.0,101.7059,10243.720934,262.094,4218.299805
2025-11-30,4218.299805,121.0527,0.057070,0.0388,6552419.0,217600.0,101.2570,10350.652186,260.724,4325.600098
2025-12-31,4325.600098,119.7456,0.044623,0.0372,6640618.0,219000.0,101.3775,10374.011335,261.358,4713.899902


In [55]:
vars_to_log = ['gold', 'dollar_index', 'fed_balance', 'labor_claims', 'ip', 'gdp','ppi', 'target_gold']
for col in vars_to_log:
    df_model[f'ln_{col}'] = np.log(df_model[col])

model_vars = ['fed_rate', 'vix'] + [f'ln_{c}' for c in vars_to_log]
df_ardl = df_model[model_vars].dropna()

df_ardl

,fed_rate,vix,ln_gold,ln_dollar_index,ln_fed_balance,ln_labor_claims,ln_ip,ln_gdp,ln_ppi,ln_target_gold
2006-02-28,0.0449,0.035934,6.330790,4.602863,13.641818,12.580219,4.587005,8.386479,5.101694,6.366127
2006-03-31,0.0459,0.033757,6.366127,4.610755,13.633599,12.617354,4.587427,8.400517,5.086361,6.479738
2006-04-30,0.0479,0.034277,6.479738,4.586407,13.646585,12.623466,4.589679,8.409050,5.088830,6.465367
2006-05-31,0.0494,0.041702,6.465367,4.582623,13.654849,12.715147,4.592711,8.415169,5.101694,6.419180
2006-06-30,0.0499,0.048840,6.419180,4.587498,13.646424,12.629705,4.592965,8.418746,5.110782,6.452364
...,...,...,...,...,...,...,...,...,...,...
2025-09-30,0.0422,0.045579,8.253436,4.788631,15.703851,12.366276,4.621287,9.224688,5.568764,8.289590
2025-10-31,0.0409,0.052211,8.289590,4.798975,15.700614,12.331603,4.622085,9.234420,5.568703,8.347187
2025-11-30,0.0388,0.057070,8.347187,4.796226,15.695345,12.290414,4.617662,9.244805,5.563462,8.372306
2025-12-31,0.0372,0.044623,8.372306,4.785369,15.708716,12.296827,4.618851,9.247059,5.565891,8.458271


In [56]:
def run_adf_test(series, name):
    # Test at Level
    result = adfuller(series.dropna())
    p_value = result[1]
    
    if p_value <= 0.05:
        return f"I(0) - Stationary (p={p_value:.4f})"
    else:
        # ถ้า Level ไม่นิ่ง ให้ลอง Test แบบ Diff (First Difference)
        diff_result = adfuller(series.diff().dropna())
        diff_p_value = diff_result[1]
        
        if diff_p_value <= 0.05:
            return f"I(1) - Stationary at Diff (p={diff_p_value:.4f})"
        else:
            return f"I(2) or Higher (Non-Stationary) (p={diff_p_value:.4f})"
        
summary_data = []
for col in df_ardl.columns:
    status = run_adf_test(df_ardl[col], col)
    summary_data.append({'Variable': col, 'Status': status})

df_status = pd.DataFrame(summary_data)
df_status

,Variable,Status
0,fed_rate,I(0) - Stationary (p=0.0220)
1,vix,I(0) - Stationary (p=0.0001)
2,ln_gold,I(1) - Stationary at Diff (p=0.0000)
3,ln_dollar_index,I(1) - Stationary at Diff (p=0.0000)
4,ln_fed_balance,I(1) - Stationary at Diff (p=0.0000)
5,ln_labor_claims,I(0) - Stationary (p=0.0042)
6,ln_ip,I(1) - Stationary at Diff (p=0.0000)
7,ln_gdp,I(1) - Stationary at Diff (p=0.0001)
8,ln_ppi,I(1) - Stationary at Diff (p=0.0000)
9,ln_target_gold,I(1) - Stationary at Diff (p=0.0000)


In [57]:
X_cols = ['fed_rate'
          ,'ln_gold'
          ,'ln_dollar_index'
          ,'vix'
          ,'ln_labor_claims'
          ,'ln_ip'
        #   ,'ln_gdp'
          ,'ln_ppi'
        #   ,'ln_fed_balance'
          ]

X = df_ardl[X_cols].dropna()
X = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data["Variable"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

vif_data

,Variable,VIF
0,const,29534.683587
1,fed_rate,1.421467
2,ln_gold,4.774124
3,ln_dollar_index,1.815856
4,vix,1.594409
5,ln_labor_claims,2.515198
6,ln_ip,2.090515
7,ln_ppi,6.020694


In [58]:
train_date_str = '2015-12-31'
df_test_ardl = df_ardl[df_ardl.index <= train_date_str].copy()
df_test_ardl

,fed_rate,vix,ln_gold,ln_dollar_index,ln_fed_balance,ln_labor_claims,ln_ip,ln_gdp,ln_ppi,ln_target_gold
2006-02-28,0.0449,0.035934,6.330790,4.602863,13.641818,12.580219,4.587005,8.386479,5.101694,6.366127
2006-03-31,0.0459,0.033757,6.366127,4.610755,13.633599,12.617354,4.587427,8.400517,5.086361,6.479738
2006-04-30,0.0479,0.034277,6.479738,4.586407,13.646585,12.623466,4.589679,8.409050,5.088830,6.465367
2006-05-31,0.0494,0.041702,6.465367,4.582623,13.654849,12.715147,4.592711,8.415169,5.101694,6.419180
2006-06-30,0.0499,0.048840,6.419180,4.587498,13.646424,12.629705,4.592965,8.418746,5.110782,6.452364
...,...,...,...,...,...,...,...,...,...,...
2015-06-30,0.0013,0.041395,7.066040,4.668324,15.318488,12.519971,4.614014,8.701684,5.264761,6.998418
2015-07-31,0.0013,0.041739,6.998418,4.689345,15.316356,12.534477,4.610864,8.707984,5.271973,7.031388
2015-08-31,0.0014,0.056084,7.031388,4.705577,15.314040,12.525253,4.617324,8.713265,5.267343,7.017058
2015-09-30,0.0014,0.070820,7.017058,4.712509,15.316051,12.504324,4.615484,8.715785,5.256974,7.040098


In [59]:
y_col = 'ln_gold'
X_cols = ['fed_rate'
          ,'ln_dollar_index'
        #   ,'vix'
          # ,'ln_labor_claims'
        #   ,'ln_ip'
        #   ,'ln_gdp'
          # ,'ln_ppi'
          # ,'ln_fed_balance'
          ]

data_ardl = df_test_ardl[[y_col] + X_cols].dropna()

custom_max_order = {
    'fed_rate': 6,
    'ln_dollar_index': 6,
    # 'vix': 3,
    # 'ln_labor_claims': 6,
    # 'ln_ip': 3,
    # 'ln_ppi': 6
}

sel_res = ardl_select_order(
    data_ardl[y_col], 
    maxlag=6, 
    exog=data_ardl[X_cols], 
    maxorder=custom_max_order,
    ic='aic'
)

print(f"Best AR Lags: {sel_res.ar_lags}")
print(f"Best DL Orders: {sel_res.dl_lags}")

Best AR Lags: [1]
Best DL Orders: {'fed_rate': [0], 'ln_dollar_index': [0, 1, 2]}


In [60]:
ar_lag = max(sel_res.ar_lags) if isinstance(sel_res.ar_lags, list) else sel_res.ar_lags
dl_lags = {k: (max(v) if isinstance(v, list) else v) for k, v in sel_res.dl_lags.items()}
exog_order = {}
for i in dl_lags:
    exog_order[i] = max(1, dl_lags[i])
    
print(f"\n--- 2. ARDL Levels Analysis & Bounds Test ---")
model_ardl = ARDL(
    data_ardl[y_col], 
    lags=ar_lag, 
    exog=data_ardl[X_cols], 
    order=exog_order
)
res_ardl = model_ardl.fit()
res_ardl.summary()


--- 2. ARDL Levels Analysis & Bounds Test ---


<class 'statsmodels.iolib.summary.Summary'>
"""
                              ARDL Model Results                              
==============================================================================
Dep. Variable:                ln_gold   No. Observations:                   85
Model:                  ARDL(1, 1, 2)   Log Likelihood                 143.263
Method:               Conditional MLE   S.D. of innovations              0.044
Date:                Fri, 20 Feb 2026   AIC                           -270.526
Time:                        03:56:41   BIC                           -251.080
Sample:                             2   HQIC                          -262.709
                                   85                                         
======================================================================================
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  1.7755      0.731      2.428      0.018       0.319       3.232
ln_gold.L1             0.9040      0.035     26.156      0.000       0.835       0.973
fed_rate.L0            2.2417      3.345      0.670      0.505      -4.419       8.902
fed_rate.L1           -3.3275      3.416     -0.974      0.333     -10.129       3.474
ln_dollar_index.L0    -1.6137      0.249     -6.491      0.000      -2.109      -1.119
ln_dollar_index.L1     1.8937      0.350      5.418      0.000       1.198       2.590
ln_dollar_index.L2    -0.5174      0.271     -1.912      0.060      -1.056       0.021
======================================================================================
"""

In [61]:
model_uecm = UECM(
    data_ardl[y_col], 
    lags=6, 
    exog=data_ardl[X_cols], 
    order=exog_order
)
res_uecm = model_uecm.fit()

# 2. รัน Bounds Test จากผลลัพธ์ของ UECM
# case 3 คือมี intercept แต่ไม่มี trend (นิยมใช้ที่สุด)
bt_results = res_uecm.bounds_test(case=3)

print("--- ARDL Bounds Test Results ---")
print(bt_results)

# 3. ดูค่า ECT (ในตาราง summary จะชื่อประมาณ 'diff.ln_gold.L1' หรือตัวแปรที่เป็นระดับ Level)
# หรือดูค่า Adjustment Term โดยตรง
print("\n--- UECM Summary (ดูค่า ECT และนัยสำคัญ) ---")
print(res_uecm.summary())

--- ARDL Bounds Test Results ---
BoundsTestResult
Stat: 2.07492
Upper P-value: 0.509
Lower P-value: 0.231
Null: No Cointegration
Alternative: Possible Cointegration


--- UECM Summary (ดูค่า ECT และนัยสำคัญ) ---
                              UECM Model Results                              
Dep. Variable:              D.ln_gold   No. Observations:                   85
Model:                  UECM(6, 1, 2)   Log Likelihood                 134.104
Method:               Conditional MLE   S.D. of innovations              7.038
Date:                Fri, 20 Feb 2026   AIC                           -242.208
Time:                        03:56:41   BIC                           -211.405
Sample:                             6   HQIC                          -229.867
                                   85                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------

## ARDL-ECM Forcast

In [62]:
exog_order_pure = {}
for i in exog_order:
    exog_order_pure[i] = [int(j) for j in range(1, exog_order[i]+1)]
ar_order = ar_lag

train_data = df_test_ardl.copy()
test_data = df_ardl[df_ardl.index > train_date_str].copy()

history = train_data.copy()
predictions = []
actuals = test_data[y_col].values

print(f"Train Period: {train_data.index[0].date()} to {train_data.index[-1].date()} (Count: {len(train_data)})")
print(f"Test Period:  {test_data.index[0].date()} to {test_data.index[-1].date()} (Count: {len(test_data)})")
print(f"\nStarting Walk-Forward Forecast (OOS)")

for t in range(len(test_data)):
    model = ARDL(
        endog=history[y_col],
        lags=ar_order,
        exog=history[X_cols],
        order=exog_order_pure,
        trend='c'
    )
    model_fit = model.fit()
    
    next_exog = test_data.iloc[[t]][X_cols]
    
    pred = model_fit.predict(start=len(history), end=len(history), exog_oos=next_exog)
    yhat = pred.values[0]
    predictions.append(yhat)
    
    history = pd.concat([history, test_data.iloc[[t]]])
    
    # warking forward
    # history = history.iloc[1:]
    
    if (t+1) % 12 == 0:
        print(f"Step {t+1}: {test_data.index[t].date()} -> Pred={np.exp(yhat):.4f} | Actual={np.exp(actuals[t]):.4f}")

final_model = ARDL(endog=history[y_col], lags=ar_order, exog=history[X_cols], order=exog_order_pure, trend='c')
final_model_fit = final_model.fit()
next_exog_future = history.iloc[[-1]][X_cols]
pred_future = final_model_fit.predict(start=len(history), end=len(history), exog_oos=next_exog_future)
yhat_future = pred_future.values[0]
# predictions.append(yhat_future)

actual_price = np.exp(actuals)
pred_price = np.exp(predictions)

results = pd.DataFrame({
    'Actual': actuals,
    'Predicted' : predictions,
    'Error' : actuals - predictions,
    'Actual_Price': actual_price,
    'Predicted_Price': pred_price,
    'Error_Price' : actual_price - pred_price
}, index=test_data.index)
last_date = results.index[-1]
next_date = last_date + BusinessDay(n=1)
future_row = pd.DataFrame({
    'Actual': [np.nan],              
    'Predicted': [yhat_future],      
    'Error': [np.nan],               
    'Actual_Price': [np.nan],        
    'Predicted_Price': [np.exp(yhat_future)], 
    'Error_Price': [np.nan]  
}, index=[next_date]) 

results = pd.concat([results, future_row])       
results.round(2)

Train Period: 2006-02-28 to 2015-12-31 (Count: 85)
Test Period:  2016-01-31 to 2026-01-31 (Count: 89)

Starting Walk-Forward Forecast (OOS)
Step 12: 2017-04-30 -> Pred=1237.8976 | Actual=1266.1000
Step 24: 2018-10-31 -> Pred=1183.0370 | Actual=1212.3000
Step 36: 2020-04-30 -> Pred=1577.0987 | Actual=1684.2000
Step 48: 2021-06-30 -> Pred=1891.8999 | Actual=1770.8000
Step 60: 2022-10-31 -> Pred=1672.0040 | Actual=1635.9000
Step 72: 2024-02-29 -> Pred=2059.5800 | Actual=2045.7000
Step 84: 2025-08-31 -> Pred=3328.2379 | Actual=3473.7000


,Actual,Predicted,Error,Actual_Price,Predicted_Price,Error_Price
2016-01-31,7.02,6.96,0.06,1116.4,1052.79,63.61
2016-02-29,7.12,7.01,0.11,1233.9,1109.37,124.53
2016-03-31,7.12,7.11,0.01,1234.2,1220.35,13.85
2016-06-30,7.18,7.10,0.08,1318.4,1211.28,107.12
2016-07-31,7.21,7.19,0.02,1349.0,1328.36,20.64
...,...,...,...,...,...,...
2025-10-31,8.29,8.27,0.02,3982.2,3905.49,76.71
2025-11-30,8.35,8.31,0.04,4218.3,4060.70,157.60
2025-12-31,8.37,8.37,0.01,4325.6,4296.21,29.39
2026-01-31,8.46,8.39,0.07,4713.9,4394.89,319.01


In [63]:
rmse = np.sqrt(mean_squared_error(actual_price, pred_price))
mae = mean_absolute_error(actual_price, pred_price)

print(f"RMSE (USD): {rmse:.2f}")
print(f"MAE (USD):  {mae:.2f}")

RMSE (USD): 95.01
MAE (USD):  66.81


In [64]:
# plt.figure(figsize=(14, 7))

# plt.axvline(x=pd.to_datetime('2015-12-31'), color='gray', linestyle=':', label='Train/Test Split')

# plt.plot(df_ardl.index, np.exp(df_ardl[y_col]), label='Actual History', color='lightgray')
# plt.plot(test_data.index, actual_price, label='Actual Test Data', color='#1f77b4', linewidth=2)
# plt.plot(test_data.index, pred_price, label='Forecast (Pure OOS)', color='#d62728', linestyle='--', linewidth=2)

# plt.title('Gold Price Forecast: Out-of-Sample Testing (2016-Present)')
# plt.xlabel('Date')
# plt.ylabel('Price (USD)')
# plt.legend()
# plt.grid(True, alpha=0.3)
# plt.show()

In [65]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_ardl.index, 
    y=np.exp(df_ardl[y_col]),
    mode='lines',
    name='Actual History',
    line=dict(color='lightgray')
))

fig.add_trace(go.Scatter(
    x=test_data.index, 
    y=actual_price,
    mode='lines',
    name='Actual Test (2016-Present)',
    line=dict(color='#1f77b4', width=2)
))

fig.add_trace(go.Scatter(
    x=test_data.index, 
    y=pred_price,
    mode='lines',
    name='Forecast',
    line=dict(color='#d62728', width=2, dash='dash')
))

fig.update_layout(
    width=1000,
    height=700,
    autosize=False,
    title='Gold Price Forecast: Interactive Walk-Forward Validation',
    yaxis_title='Price (USD)',
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(count=5, label="5y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(visible=True),
        type="date"
    ),
    template="plotly_white",
    legend=dict(x=0, y=1)
)

fig.show()

In [66]:
exog_order_pure = {}
for i in exog_order:
    exog_order_pure[i] = [int(j) for j in range(1, exog_order[i]+1)]
ar_order = ar_lag

# --- 1. Define Stationary and Non-Stationary Variables ---
# Based on your ADF test results
i0_vars = ['vix', 'ln_labor_claims'] 
i1_vars = [col for col in df_ardl.columns if col not in i0_vars]

# --- 2. Create a Stationary DataFrame for ARDL modeling ---
df_stat = df_ardl.copy()
df_stat[i1_vars] = df_stat[i1_vars].diff() # Apply First Difference to I(1) variables ONLY
df_stat = df_stat.dropna() # Drop the first row containing NaNs from differencing

# --- 3. Split Data into Train and Test ---
# 'stat' is used for model training/predicting (differences)
train_data_stat = df_stat[df_stat.index <= train_date_str].copy()
test_data_stat  = df_stat[df_stat.index > train_date_str].copy()

# 'level' is kept for price reconstruction later
history_level = df_ardl[df_ardl.index <= train_date_str].copy()
test_data_level = df_ardl[df_ardl.index > train_date_str].copy()

history_stat = train_data_stat.copy()

predictions_diff = []
predictions_log_level = []
actuals_log_level = test_data_level[y_col].values

print(f"Train Period: {train_data_stat.index[0].date()} to {train_data_stat.index[-1].date()} (Count: {len(train_data_stat)})")
print(f"Test Period:  {test_data_stat.index[0].date()} to {test_data_stat.index[-1].date()} (Count: {len(test_data_stat)})")
print(f"\nStarting Walk-Forward Forecast (Stationary OOS)")

# --- 4. Walk-Forward Loop ---
for t in range(len(test_data_stat)):
    
    # Fit ARDL on Stationary Data
    model = ARDL(
        endog=history_stat[y_col],
        lags=ar_order,
        exog=history_stat[X_cols],
        order=exog_order_pure,
        trend='c'
    )
    model_fit = model.fit()
    
    # Predict the next difference (Return)
    next_exog = test_data_stat.iloc[[t]][X_cols]
    pred_diff = model_fit.predict(start=len(history_stat), end=len(history_stat), exog_oos=next_exog).values[0]
    
    # Reconstruct Log Level: Previous Actual Log Level + Predicted Difference
    prev_actual_log_level = history_level[y_col].iloc[-1]
    yhat_log_level = prev_actual_log_level + pred_diff
    
    predictions_diff.append(pred_diff)
    predictions_log_level.append(yhat_log_level)
    
    # Update histories with the new actual row
    history_stat = pd.concat([history_stat, test_data_stat.iloc[[t]]])
    history_level = pd.concat([history_level, test_data_level.iloc[[t]]])
    
    if (t+1) % 12 == 0:
        print(f"Step {t+1}: {test_data_stat.index[t].date()} -> Pred Price={np.exp(yhat_log_level):.2f} | Actual Price={np.exp(actuals_log_level[t]):.2f}")

# --- 5. Predict the Future (T+1) ---
final_model = ARDL(
    endog=history_stat[y_col], 
    lags=ar_order, 
    exog=history_stat[X_cols], 
    order=exog_order_pure, 
    trend='c'
)
final_model_fit = final_model.fit()

# Use the last known exog values (Naive approach) for future prediction
next_exog_future = history_stat.iloc[[-1]][X_cols] 
pred_diff_future = final_model_fit.predict(start=len(history_stat), end=len(history_stat), exog_oos=next_exog_future).values[0]

# Reconstruct future price
prev_actual_log_level_future = history_level[y_col].iloc[-1]
yhat_log_level_future = prev_actual_log_level_future + pred_diff_future

# --- 6. Assemble the Final Results DataFrame & Apply mcal ---
actual_price = np.exp(actuals_log_level)
pred_price = np.exp(predictions_log_level)

results = pd.DataFrame({
    'Actual': actuals_log_level,
    'Predicted' : predictions_log_level,
    'Error' : actuals_log_level - predictions_log_level,
    'Actual_Price': actual_price,
    'Predicted_Price': pred_price,
    'Error_Price' : actual_price - pred_price
}, index=test_data_stat.index)

last_date = results.index[-1]

# Use mcal to find the exact next CME Gold trading day
market_cal = mcal.get_calendar('CMEGlobex_Gold')
start_search = last_date + pd.Timedelta(days=1)
end_search = start_search + pd.Timedelta(days=15) # Buffer for long holidays

# Get the exact next valid trading day
valid_days = market_cal.valid_days(start_date=start_search, end_date=end_search)
next_date = valid_days[0].tz_localize(None) # Remove timezone for clean index

# Append the future prediction row
future_row = pd.DataFrame({
    'Actual': [np.nan],              
    'Predicted': [yhat_log_level_future],                    
    'Actual_Price': [np.nan],        
    'Predicted_Price': [np.exp(yhat_log_level_future)], 
    'Error': [np.nan],
    'Error_Price': [np.nan]  
}, index=[next_date]) 

results = pd.concat([results, future_row])       

print("\nFinal Results (with exact CME Trading Dates):")
results.round(2)

Train Period: 2006-03-31 to 2015-12-31 (Count: 84)
Test Period:  2016-01-31 to 2026-01-31 (Count: 89)

Starting Walk-Forward Forecast (Stationary OOS)
Step 12: 2017-04-30 -> Pred Price=1252.76 | Actual Price=1266.10
Step 24: 2018-10-31 -> Pred Price=1190.06 | Actual Price=1212.30
Step 36: 2020-04-30 -> Pred Price=1605.57 | Actual Price=1684.20
Step 48: 2021-06-30 -> Pred Price=1909.91 | Actual Price=1770.80
Step 60: 2022-10-31 -> Pred Price=1665.89 | Actual Price=1635.90
Step 72: 2024-02-29 -> Pred Price=2072.20 | Actual Price=2045.70
Step 84: 2025-08-31 -> Pred Price=3326.98 | Actual Price=3473.70

Final Results (with exact CME Trading Dates):


,Actual,Predicted,Error,Actual_Price,Predicted_Price,Error_Price
2016-01-31,7.02,6.97,0.05,1116.4,1066.18,50.22
2016-02-29,7.12,7.01,0.11,1233.9,1109.22,124.68
2016-03-31,7.12,7.11,0.01,1234.2,1221.43,12.77
2016-06-30,7.18,7.12,0.07,1318.4,1232.89,85.51
2016-07-31,7.21,7.20,0.01,1349.0,1339.65,9.35
...,...,...,...,...,...,...
2025-10-31,8.29,8.26,0.02,3982.2,3885.13,97.07
2025-11-30,8.35,8.30,0.04,4218.3,4037.63,180.67
2025-12-31,8.37,8.36,0.01,4325.6,4274.92,50.68
2026-01-31,8.46,8.39,0.07,4713.9,4381.45,332.45


In [67]:
rmse = np.sqrt(mean_squared_error(actual_price, pred_price))
mae = mean_absolute_error(actual_price, pred_price)

print(f"RMSE (USD): {rmse:.2f}")
print(f"MAE (USD):  {mae:.2f}")

RMSE (USD): 97.90
MAE (USD):  68.45


In [68]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_ardl.index, 
    y=np.exp(df_ardl[y_col]),
    mode='lines',
    name='Actual History',
    line=dict(color='lightgray')
))
fig.add_trace(go.Scatter(
    x=test_data_stat.index,
    y=actual_price,
    mode='lines',
    name='Actual Test (2016-Present)',
    line=dict(color='#1f77b4', width=2)
))
fig.add_trace(go.Scatter(
    x=test_data_stat.index,
    y=pred_price,
    mode='lines',
    name='Forecast',
    line=dict(color='#d62728', width=2, dash='dash')
))
fig.update_layout(
    width=1000,
    height=700,
    autosize=False,
    title='Gold Price Forecast: Walk-Forward Validation with Stationary ARDL',
    yaxis_title='Price (USD)',
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(count=5, label="5y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(visible=True),
        type="date"
    ),
    template="plotly_white",
    legend=dict(x=0, y=1)
)
fig.show()

## Technical Data

In [69]:
def denoise_data(data, wavelet='db4', level=2):
    coeff = pywt.wavedec(data, wavelet, mode="per")
    sigma = (1/0.6745) * np.median(np.abs(coeff[-1] - np.median(coeff[-1])))
    uthesh = sigma * np.sqrt(2 * np.log(len(data)))
    new_coeff = [coeff[0]]
    for i in coeff[1:]:
        new_coeff.append(pywt.threshold(i, value=uthesh, mode='soft'))
    reconstructed = pywt.waverec(new_coeff, wavelet, mode='per')
    return reconstructed[:len(data)]

In [70]:
macro_feature = results[['Predicted']].copy()
macro_feature.columns = ['Macro_Signal']

# Resample monthly to daily and forward fill
macro_daily = macro_feature.resample('D').asfreq()
macro_daily = macro_daily.ffill()

# Normalize datetime index (remove time component)
macro_daily.index = pd.to_datetime(macro_daily.index.date)
macro_daily

,Macro_Signal
2016-01-31,6.971841
2016-02-01,6.971841
2016-02-02,6.971841
2016-02-03,6.971841
2016-02-04,6.971841
...,...
2026-01-29,8.360520
2026-01-30,8.360520
2026-01-31,8.385136
2026-02-01,8.385136


In [82]:
df_daily = pd.read_csv('all_gold_data.csv', index_col=0, parse_dates=True)
df_daily.sort_index(inplace=True)
df_daily = df_daily[~df_daily.index.duplicated(keep='first')]

# Normalize datetime index (remove time component)
df_daily.index = pd.to_datetime(df_daily.index.date)

df_daily['actual_close'] = df_daily['close'].copy()
df_daily['ln_close'] = np.log(df_daily['close'])

for col in ['open', 'high', 'low', 'close']:
    df_daily[col] = denoise_data(df_daily[col].values)

# Momentum & Trend
df_daily.ta.rsi(length=14, append=True)
df_daily.ta.macd(fast=12, slow=26, signal=9, append=True)
df_daily.ta.adx(length=14, append=True)
df_daily.ta.cci(length=20, append=True)

# Volatility & Bands
df_daily.ta.bbands(length=20, std=2, append=True)
df_daily.ta.atr(length=14, append=True)

# Moving Average Distances
df_daily.ta.ema(length=12, append=True)
df_daily.ta.ema(length=24, append=True)
df_daily['dist_ema12'] = (df_daily['close'] - df_daily['EMA_12']) / df_daily['EMA_12']
df_daily['dist_ema24'] = (df_daily['close'] - df_daily['EMA_24']) / df_daily['EMA_24']

# Statistical & Others
df_daily['daily_range'] = (df_daily['high'] - df_daily['low']) / df_daily['open']
rolling_mean = df_daily['close'].rolling(window=20).mean()
rolling_std = df_daily['close'].rolling(window=20).std()
df_daily['z_score'] = (df_daily['close'] - rolling_mean) / rolling_std

cols_to_drop = ['EMA_50', 'EMA_200', 'BBU_20_2.0', 'BBL_20_2.0', 'BBM_20_2.0']
df_daily.drop(columns=[c for c in cols_to_drop if c in df_daily.columns], inplace=True)
df_daily.dropna(inplace=True)
df_daily

,open,high,low,close,volume,actual_close,ln_close,RSI_14,MACD_12_26_9,MACDh_12_26_9,...,CCI_20_0.015,BBB_20_2.0,BBP_20_2.0,ATRr_14,EMA_12,EMA_24,dist_ema12,dist_ema24,daily_range,z_score
2009-05-01,897.250011,936.557010,852.232076,898.404571,234559,978.799988,6.886327,63.270062,57.873442,-11.821603,...,83.511104,25.722203,0.778890,103.291859,839.885931,790.242674,0.069675,0.136872,0.093982,1.087314
2009-06-01,924.417372,955.477683,895.015665,931.939695,12773,927.099976,6.832061,66.435254,60.929840,-7.012164,...,117.579535,25.053580,0.896989,99.964803,854.048049,801.578435,0.091203,0.162631,0.065406,1.547748
2009-07-01,949.580444,977.419087,929.395823,961.151437,177420,953.700012,6.860349,68.945662,64.960379,-2.385300,...,146.205513,25.581698,0.972329,95.955013,870.525493,814.344276,0.104105,0.180277,0.050573,1.841477
2009-08-01,974.900457,1005.265673,957.240079,988.231734,7492,951.700012,6.858250,71.103352,69.538170,1.753993,...,162.757146,27.590919,1.006014,92.276177,888.634146,828.255272,0.112079,0.193149,0.049262,1.972807
2009-09-01,998.726419,1033.930154,980.815596,1012.887785,15642,1008.000000,6.915723,72.946440,74.299166,5.211991,...,170.229644,30.353602,1.011015,89.286242,907.750090,843.025873,0.115822,0.201491,0.053182,1.992303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-01,3749.976879,4266.472847,3722.104343,3924.175197,42621,3982.199951,8.289590,99.262639,461.660075,112.282725,...,158.244673,84.288084,0.980573,228.347764,3252.370723,2839.953757,0.206558,0.381774,0.145166,1.873620
2025-11-01,3922.375670,4233.189166,3891.102609,4111.843805,194787,4218.299805,8.347187,99.364265,496.024988,117.318110,...,148.232308,86.857349,0.983474,236.471966,3384.597351,2941.704961,0.214869,0.397776,0.087214,1.884930
2025-12-01,4092.656979,4508.472438,4062.516147,4345.179073,21215,4325.600098,8.372306,99.463308,535.909998,125.762496,...,151.589888,89.657337,0.995397,251.435136,3532.379155,3053.982890,0.230100,0.422791,0.108965,1.931411
2026-01-01,4394.418032,5400.759516,4345.878342,4600.375998,159644,4713.899902,8.458271,99.546520,581.409290,137.009431,...,175.201195,92.703205,1.005441,308.874101,3696.686361,3177.694339,0.244459,0.447709,0.240050,1.970572


In [83]:
"""
For joining daily-marco variable to technical data
"""

macro_to_merge = dollar_index.join(vix, how='inner')
macro_to_merge = macro_to_merge[macro_to_merge.index >= '2016-02-01']
macro_to_merge

,Dollar Index,VIX
2016-02-01,115.3331,0.057677
2016-02-02,115.5530,0.063451
2016-02-03,114.6144,0.062498
2016-02-04,113.6012,0.063047
2016-02-05,114.2935,0.067492
...,...,...
2026-02-09,117.6392,0.050114
2026-02-10,117.5216,0.051355
2026-02-11,117.4601,0.050951
2026-02-12,117.5376,0.060102


In [84]:
df_final = macro_daily.join(df_daily, how='right')
df_final = df_final.join(macro_to_merge, how='left')
df_final = df_final.ffill()
df_final = df_final.dropna()
forecast_horizon = 1
for i in range(1, forecast_horizon + 1):
    col_name = f'target_return_{i}m'
    df_final[col_name] = np.log(df_final['actual_close']).shift(-i) - np.log(df_final['close'])

df_predict_latest = df_final.tail(forecast_horizon).copy()

# threshold = 0.00
# choices = [1, 0]
# for i in range(1, forecast_horizon + 1):
#     target_col = f'target_return_{i}m'
#     signal_col = f'signal_{i}m'
    
#     conditions = [
#         (df_final[target_col] >= threshold),
#         (df_final[target_col] < -threshold)
#     ]
    
#     df_final[signal_col] = np.select(conditions, choices, default=0)


df_final.to_csv('gold_technical.csv', index=True)
df_final

,Macro_Signal,open,high,low,close,volume,actual_close,ln_close,RSI_14,MACD_12_26_9,...,ATRr_14,EMA_12,EMA_24,dist_ema12,dist_ema24,daily_range,z_score,Dollar Index,VIX,target_return_1m
2016-02-01,6.971841,1174.299360,1234.167050,1148.932786,1207.396135,13930,1233.900024,7.117935,36.371736,-49.413501,...,90.515438,1195.347144,1238.173265,0.010080,-0.024857,0.072583,0.296514,115.3331,0.057677,0.021957
2016-03-01,7.011414,1186.880527,1248.212064,1165.107802,1216.858135,244751,1234.199951,7.118178,40.177220,-44.037265,...,89.985812,1198.656527,1236.468055,0.015185,-0.015860,0.070019,0.779092,113.9265,0.051095,0.057750
2016-04-01,7.107779,1201.270879,1264.344311,1183.871693,1228.489160,9954,1289.199951,7.161777,44.566112,-38.395434,...,89.305993,1203.246163,1235.829743,0.020979,-0.005940,0.066990,1.422708,110.2977,0.037816,0.070634
2016-06-01,7.107779,1218.453498,1283.268988,1206.631344,1240.510926,13380,1318.400024,7.184174,48.751221,-32.578640,...,88.400733,1208.979203,1236.204238,0.026081,0.003484,0.062897,1.978199,112.4620,0.040992,0.083840
2016-07-01,7.117114,1238.271698,1304.876483,1233.235326,1253.388434,281653,1349.000000,7.207119,52.857011,-26.622791,...,87.203156,1215.811393,1237.578973,0.030907,0.012775,0.057856,2.222366,111.6509,0.042637,0.041807
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-01,8.160846,3749.976879,4266.472847,3722.104343,3924.175197,42621,3982.199951,8.289590,99.262639,461.660075,...,228.347764,3252.370723,2839.953757,0.206558,0.381774,0.145166,1.873620,120.1502,0.047025,0.072276
2025-11-01,8.264911,3922.375670,4233.189166,3891.102609,4111.843805,194787,4218.299805,8.347187,99.364265,496.024988,...,236.471966,3384.597351,2941.704961,0.214869,0.397776,0.087214,1.884930,120.1502,0.047025,0.050679
2025-12-01,8.303413,4092.656979,4508.472438,4062.516147,4345.179073,21215,4325.600098,8.372306,99.463308,535.909998,...,251.435136,3532.379155,3053.982890,0.230100,0.422791,0.108965,1.931411,120.9862,0.049768,0.081449
2026-01-01,8.360520,4394.418032,5400.759516,4345.878342,4600.375998,159644,4713.899902,8.458271,99.546520,581.409290,...,308.874101,3696.686361,3177.694339,0.244459,0.447709,0.240050,1.970572,119.7456,0.043157,0.087352


In [85]:
df_final.columns

Index(['Macro_Signal', 'open', 'high', 'low', 'close', 'volume',
       'actual_close', 'ln_close', 'RSI_14', 'MACD_12_26_9', 'MACDh_12_26_9',
       'MACDs_12_26_9', 'ADX_14', 'DMP_14', 'DMN_14', 'CCI_20_0.015',
       'BBB_20_2.0', 'BBP_20_2.0', 'ATRr_14', 'EMA_12', 'EMA_24', 'dist_ema12',
       'dist_ema24', 'daily_range', 'z_score', 'Dollar Index', 'VIX',
       'target_return_1m'],
      dtype='object')

## Unique

In [ ]:
# --- ⚙️ Config ---
N_MODELS = 10
WINDOW_SIZE = 6      # มองย้อนหลัง 6 เดือน
BATCH_SIZE = 8       
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_DIR = 'ensemble_huber_monthly' # 📁 เปลี่ยนชื่อโฟลเดอร์กันไปทับของเดิม


def set_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    tf.random.set_seed(seed)
    np.random.seed(seed)

print("🚀 Loading Monthly Data...")

# --- 1. Data Preparation ---
# สมมติว่าไฟล์รายเดือนของคุณชื่อ gold_technical.csv (ถ้าเปลี่ยนชื่อไฟล์ อย่าลืมแก้ตรงนี้นะครับ)
df = pd.read_csv('gold_technical.csv', index_col=0, parse_dates=True)

# ตัดบรรทัดสุดท้ายทิ้ง (เพราะเรายังไม่รู้ค่าของอนาคต 1 เดือนข้างหน้า)
# df = df[:-1] 

# ดรอปแถวที่มี NaN ในคอลัมน์เป้าหมาย
df_train_full = df.dropna(subset=['target_return_1m']).copy()

# เลือก Features (ตัดเป้าหมายและราคาจริงออก)
feature_cols = [c for c in df.columns if 'target' not in c and c != 'actual_close' and c != 'close']
target_cols = ['target_return_1m'] # 🎯 เป้าหมายเดียว: ผลตอบแทน 1 เดือน

X = df_train_full[feature_cols].values
y = df_train_full[target_cols].values

# แบ่ง Train 80% / Test 20%
train_size = int(len(X) * 0.8)
X_train_raw, X_test_raw = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Scaling
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train_raw)
X_test_scaled = scaler_X.transform(X_test_raw)

def create_sequences(X, y, window_size):
    Xs, ys = [], []
    for i in range(len(X) - window_size):
        Xs.append(X[i:(i + window_size)])
        ys.append(y[i + window_size])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, WINDOW_SIZE)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test, WINDOW_SIZE)

# --- 2. Multi-Output Ensemble Training ---
print(f"\n🧠 Training Huber Ensemble for Match 6 ({N_MODELS} Models)...")
model_files = []

for i in range(N_MODELS):
    print(f"   Training Model {i+1}/{N_MODELS}...")
    set_seeds(42 + i) 
    
    model = Sequential([
        Conv1D(filters=16, kernel_size=1, padding='same', activation='swish', input_shape=(WINDOW_SIZE, len(feature_cols))),
        MaxPooling1D(pool_size=1),
        Bidirectional(LSTM(32, return_sequences=False, activation='tanh')),
        Dropout(0.3),
        Dense(16, activation='swish'),
        Dense(1) # 🎯 เปลี่ยนเป็น Dense(1) ทายจุดเดียว (รายเดือน)
    ])

    model.compile(optimizer=Adam(learning_rate=0.001), loss=Huber(), metrics=['mae'])
    
    filename = os.path.join(MODEL_DIR, f'gold_monthly_ens_{i}.keras')
    model_files.append(filename)
    
    callbacks = [
        ModelCheckpoint(filename, save_best_only=True, monitor='val_loss', mode='min', verbose=0),
        EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)
    ]

    model.fit(
        X_train_seq, y_train_seq,
        epochs=100, 
        batch_size=BATCH_SIZE,
        validation_data=(X_test_seq, y_test_seq),
        callbacks=callbacks,
        verbose=0
    )

print(f"\n✅ All ensemble models saved in '{MODEL_DIR}/'")

# --- 3. Prediction For Next Month ---
print(f"\n🔮 Generating Next-Month Forecast...")

# ดึงข้อมูล 10 เดือนล่าสุด
last_window = df[feature_cols].tail(WINDOW_SIZE).values
last_window_scaled = scaler_X.transform(last_window).reshape(1, WINDOW_SIZE, len(feature_cols))

all_preds = []
for filename in model_files:
    m = tf.keras.models.load_model(filename)
    all_preds.append(m.predict(last_window_scaled, verbose=0)[0][0]) # Extract single value

avg_return = np.mean(all_preds)
current_actual_price = df['actual_close'].iloc[-1]
last_date = df.index[-1]

# คำนวณวันที่ของเดือนถัดไป
next_month_date = last_date + pd.DateOffset(months=1)
predicted_price = current_actual_price * np.exp(avg_return)

print("\n" + "="*55)
print(f"📅 Latest Closing Price ({last_date.strftime('%Y-%m')}): {current_actual_price:.2f}")
print("🏆 Final Ensemble Forecast (Match 6):")
print("-" * 55)
trend = "🟢 UP" if predicted_price > current_actual_price else "🔴 DOWN"
print(f"🎯 Forecast End of {next_month_date.strftime('%Y-%m')}: {predicted_price:.2f}  {trend} ({predicted_price - current_actual_price:+.2f})")
print("="*55)

# --- 4. Final Accuracy Evaluation (RMSE) ---
print(f"\n📊 Calculating Final Test RMSE...")
test_preds_all = []
for filename in model_files:
    m = tf.keras.models.load_model(filename)
    test_preds_all.append(m.predict(X_test_seq, verbose=0))

# บีบมิติ Array ให้เป็น 1D
y_pred_avg = np.mean(test_preds_all, axis=0).flatten() 
test_start_idx = train_size + WINDOW_SIZE

base_prices = df_train_full['close'].iloc[test_start_idx:].values[:len(y_pred_avg)]
y_test_flat = y_test_seq[:len(base_prices)].flatten()

true_p = base_prices * np.exp(y_test_flat)
pred_p = base_prices * np.exp(y_pred_avg)
rmse = np.sqrt(mean_squared_error(true_p, pred_p))

print("\n" + "="*50)
print(f"📊 Accuracy Summary (Monthly Ensemble)")
print("="*50)
print(f"📉 RMSE (1-Month Ahead) : ${rmse:.2f}")
print("="*50)

🚀 Loading Monthly Data...

🧠 Training Huber Ensemble for Match 6 (10 Models)...
   Training Model 1/10...
   Training Model 2/10...
   Training Model 3/10...
   Training Model 4/10...
   Training Model 5/10...
   Training Model 6/10...
   Training Model 7/10...
   Training Model 8/10...
   Training Model 9/10...
   Training Model 10/10...

✅ All ensemble models saved in 'ensemble_huber_monthly/'

🔮 Generating Next-Month Forecast...

📅 Latest Closing Price (2026-01): 4713.90
🏆 Final Ensemble Forecast (Match 6):
-------------------------------------------------------
🎯 Forecast End of 2026-02: 4803.08  🟢 UP (+89.18)

📊 Calculating Final Test RMSE...

📊 Accuracy Summary (Monthly Ensemble)
📉 RMSE (1-Month Ahead) : $130.53


## Evaluation

In [95]:
# --- 5. MATCH 6 VAAE EVALUATION ---
print(f"\n🏆 Calculating VAAE (Match 6 Rules: 36-Month Sigma)...")

# กติกา Match 6: ใช้ Standard Deviation ของราคาปิดย้อนหลัง 36 เดือน
sigma_36m_rolling = df['actual_close'].rolling(window=36).std()

# ดึงค่า Sigma ให้ตรงกับช่วง Test Set
test_sigmas = sigma_36m_rolling.iloc[test_start_idx : test_start_idx + len(y_pred_avg)].values

# คำนวณ Absolute Error
abs_error = np.abs(pred_p - true_p)

# คำนวณ VAAE (ระวังกรณีข้อมูลสั้นเกินไปจน Sigma ช่วงแรกเป็น NaN)
vaae_results = abs_error / test_sigmas

# สร้าง DataFrame เพื่อสรุปผล
test_dates = df_train_full.index[test_start_idx : test_start_idx + len(y_pred_avg)]
vaae_df = pd.DataFrame({
    'Actual_Price': true_p,
    'Predicted_Price': pred_p,
    'Abs_Error': abs_error,
    'Sigma_36m': test_sigmas,
    'VAAE': vaae_results
}, index=test_dates)

# ตัดแถวที่ Sigma เป็น NaN ทิ้ง (กรณีข้อมูลอดีตไม่ถึง 36 เดือนตอนคำนวณ)
vaae_df_clean = vaae_df.dropna()

print("\n" + "="*65)
print("📊 MATCH 6: HISTORICAL VAAE PERFORMANCE REPORT (TEST SET)")
print("="*65)
if len(vaae_df_clean) > 0:
    overall_vaae = vaae_df_clean['VAAE'].mean()
    print(f"🏆 OVERALL TEST SET VAAE SCORE: {overall_vaae:.4f}")
    print("   (Lower is better | Goal: < 1.0)")
    print("-" * 65)
    print("\n🔍 Latest VAAE Samples:")
else:
    print("⚠️ Warning: Not enough data to calculate 36-month rolling sigma.")
print("="*65)

print(vaae_df_clean.tail(5))


🏆 Calculating VAAE (Match 6 Rules: 36-Month Sigma)...

📊 MATCH 6: HISTORICAL VAAE PERFORMANCE REPORT (TEST SET)
🏆 OVERALL TEST SET VAAE SCORE: 0.2386
   (Lower is better | Goal: < 1.0)
-----------------------------------------------------------------

🔍 Latest VAAE Samples:
            Actual_Price  Predicted_Price   Abs_Error   Sigma_36m      VAAE
2025-09-01   3982.199951      3928.755614   53.444337  588.702272  0.090783
2025-10-01   4218.299805      4049.609186  168.690619  646.975133  0.260737
2025-11-01   4325.600098      4213.047240  112.552857  709.133771  0.158719
2025-12-01   4713.899902      4502.991190  210.908712  766.202981  0.275265
2026-01-01   5020.299805      4754.587654  265.712151  837.504575  0.317267
